## Preprocessing

Input: `data/processed/train.parquet`, `data/processed/test.parquet` (saved từ `EDA.ipynb`)

Output: `data/processed/X_train.parquet`, `X_test.parquet`, `y_train.parquet`, `y_test.parquet`

Các bước:
1. Fill NaN trong dynamic features (VLE + assessment)
2. Impute `imd_band = '?'` theo phân phối từ train
3. Encode categorical features
4. Lưu feature matrix

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))
from config import DB_PATH, SNAPSHOTS, STATIC_FEATURES, RANDOM_SEED

import numpy as np
import pandas as pd
import sqlite3
import warnings
warnings.filterwarnings('ignore')

In [2]:
train = pd.read_parquet('../data/processed/train.parquet')
test  = pd.read_parquet('../data/processed/test.parquet')

# Cần assessment để tính lại num_due trong fill_in_assessment
conn = sqlite3.connect(DB_PATH)
assessment = pd.read_sql("SELECT * FROM assessments", conn)
conn.close()

print('train:', train.shape)
print('test: ', test.shape)

train: (142384, 22)
test:  (35910, 22)


---

### 1. Fill NaN — Dynamic features

In [4]:
from src.features.build_features import (
    fill_in_weekly_clicks, fill_in_assessment,
    fit_imd_distributions, transform_imd_imputation,
    DISAB_MAP, EDU_ORDER, IMD_ORDER,
)

train = fill_in_weekly_clicks(train)
train = fill_in_assessment(train, assessment)

test = fill_in_weekly_clicks(test)
test = fill_in_assessment(test, assessment)

In [5]:
# avg_score_filled: -1 sentinel (ngoài range [0,100] — phân biệt với điểm 0 thực)
# avg_days_early_filled: 0 (no_submission_despite_due đã capture trường hợp không nộp)
for df in [train, test]:
    df['avg_days_early_filled'] = df['avg_days_early'].fillna(0)
    df['avg_score_filled']      = df['avg_score'].fillna(-1)

dynamic_features = [
    'total_clicks_filled', 'active_weeks_filled', 'avg_weekly_clicks_filled',
    'num_due', 'num_submitted_filled', 'avg_score_filled',
    'num_failed_filled', 'avg_days_early_filled', 'no_submission_despite_due',
]

null_counts = pd.DataFrame({
    'train': train[dynamic_features].isna().sum(),
    'test':  test[dynamic_features].isna().sum(),
})
print(null_counts[null_counts.any(axis=1)])

Empty DataFrame
Columns: [train, test]
Index: []


In [6]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 142384 entries, 0 to 142383
Data columns (total 31 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   id_student                 142384 non-null  int64  
 1   code_module                142384 non-null  object 
 2   code_presentation          142384 non-null  object 
 3   gender                     142384 non-null  object 
 4   region                     142384 non-null  object 
 5   age_band                   142384 non-null  object 
 6   imd_band                   142384 non-null  object 
 7   highest_education          142384 non-null  object 
 8   disability                 142384 non-null  object 
 9   num_of_prev_attempts       142384 non-null  int64  
 10  studied_credits            142384 non-null  int64  
 11  date_registration          142384 non-null  object 
 12  date_unregistration        12920 non-null   float64
 13  total_clicks               13

---

### 2. Impute `imd_band = '?'`

Trước khi impute, kiểm định thống kê để xác định cơ chế missing (MCAR / MAR / MNAR):
- **MCAR / MAR** → tạo indicator + impute theo phân phối từng region
- **MNAR** → chỉ tạo indicator, fill bằng mode (impute theo region sẽ sai lệch)

In [7]:
from scipy import stats as scipy_stats

# Dùng static snapshot để test — mỗi student-course 1 dòng (imd_band không đổi theo snapshot)
train_static = train.drop_duplicates(subset=['id_student', 'code_module', 'code_presentation']).copy()
train_static['imd_missing'] = (train_static['imd_band'] == '?').astype(int)

n_miss = train_static['imd_missing'].sum()
print(f"imd_band '?' — {n_miss:,} / {len(train_static):,} students ({n_miss / len(train_static) * 100:.2f}%)")
print()

# ── MCAR proxy: Mann-Whitney U ──────────────────────────────────────────────
print('[MCAR proxy] Mann-Whitney U — biến số')
mcar_rows = []
for col in ['num_of_prev_attempts', 'studied_credits']:
    g_miss = train_static.loc[train_static['imd_missing'] == 1, col]
    g_obs  = train_static.loc[train_static['imd_missing'] == 0, col]
    _, p = scipy_stats.mannwhitneyu(g_miss, g_obs, alternative='two-sided')
    mcar_rows.append({'Biến': col, 'p-value': round(p, 4),
                      'Kết luận': '❌ Không MCAR' if p < 0.05 else '✅ MCAR'})
display(pd.DataFrame(mcar_rows))
n_mcar_reject = sum(r['p-value'] < 0.05 for r in mcar_rows)

# ── MAR: Chi-square ─────────────────────────────────────────────────────────
print('\n[MAR] Chi-square — biến categorical')
mar_rows = []
for col in ['region', 'gender', 'highest_education', 'disability', 'final_result', 'code_module']:
    ct = pd.crosstab(train_static[col], train_static['imd_missing'])
    if ct.shape[1] < 2:
        continue
    chi2, p, dof, _ = scipy_stats.chi2_contingency(ct)
    n = ct.sum().sum()
    v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
    mar_rows.append({'Biến': col, 'p-value': round(p, 4),
                     "Cramér's V": round(v, 4),
                     'Kết luận': '❌ MAR' if p < 0.05 else '✅ Độc lập'})
df_mar = pd.DataFrame(mar_rows).sort_values("Cramér's V", ascending=False)
display(df_mar)
n_mar_sig = (df_mar['p-value'] < 0.05).sum()

# ── Kết luận ────────────────────────────────────────────────────────────────
is_mcar = n_mcar_reject == 0
is_mar  = (not is_mcar) and (n_mar_sig > 0)

print()
if is_mcar:
    print('→ MCAR: impute theo phân phối tổng thể + thêm indicator')
elif is_mar:
    top_var, top_v = df_mar.iloc[0]['Biến'], df_mar.iloc[0]["Cramér's V"]
    print(f'→ MAR ({n_mar_sig}/{len(df_mar)} biến liên quan | mạnh nhất: {top_var}, V={top_v})')
    print('   → Impute theo phân phối từng region + thêm indicator')
else:
    print('→ Có thể MNAR — chỉ tạo indicator, fill bằng mode')

imd_band '?' — 817 / 19,988 students (4.09%)

[MCAR proxy] Mann-Whitney U — biến số


,Biến,p-value,Kết luận
0,num_of_prev_attempts,0.0003,❌ Không MCAR
1,studied_credits,0.0031,❌ Không MCAR



[MAR] Chi-square — biến categorical


,Biến,p-value,Cramér's V,Kết luận
0,region,0.0,0.5768,❌ MAR
2,highest_education,0.0,0.2143,❌ MAR
5,code_module,0.0,0.1011,❌ MAR
4,final_result,0.0,0.0690,❌ MAR
1,gender,0.0,0.0652,❌ MAR
3,disability,0.0,0.0439,❌ MAR



→ MAR (6/6 biến liên quan | mạnh nhất: region, V=0.5768)
   → Impute theo phân phối từng region + thêm indicator


In [8]:
# Tạo indicator trước khi impute — luôn thực hiện bất kể cơ chế missing
for df in [train, test]:
    df['imd_missing'] = (df['imd_band'] == '?').astype(int)

print(f"imd_missing — train: {train['imd_missing'].sum():,} | test: {test['imd_missing'].sum():,}")
print()

if is_mar or is_mcar:
    # MAR/MCAR: impute theo phân phối của từng region (fit trên train, áp dụng cho cả test)
    imd_distributions = fit_imd_distributions(train)
    rng = np.random.default_rng(RANDOM_SEED)
    train = transform_imd_imputation(train, imd_distributions, rng)
    test  = transform_imd_imputation(test,  imd_distributions, rng)
    n_remain_train = (train['imd_band_filled'] == '?').sum()
    n_remain_test  = (test['imd_band_filled'] == '?').sum()
    print(f"Sau impute — imd_band_filled '?' còn lại: train={n_remain_train} | test={n_remain_test}")
else:
    # MNAR: không impute theo region (sẽ sai lệch) — fill bằng mode toàn cục
    # imd_missing indicator đã capture tín hiệu missingness
    mode_imd = train.loc[train['imd_band'] != '?', 'imd_band'].mode()[0]
    for df in [train, test]:
        df['imd_band_filled'] = df['imd_band'].replace('?', mode_imd)
    print(f"MNAR: fill imd_band_filled '?' bằng mode='{mode_imd}' + giữ indicator")

imd_missing — train: 5,986 | test: 1,497

Sau impute — imd_band_filled '?' còn lại: train=0 | test=0


---

### 3. Encode categorical features

| Feature | Kiểu | Lý do |
|---|---|---|
| `disability` | Binary | Y→1, N→0 |
| `highest_education` | Ordinal | Có thứ tự theo bậc học |
| `imd_band_filled` | Ordinal | Có thứ tự theo mức độ nghèo |

> `gender`, `region`, `age_band`, `prediction_point` bị loại khỏi feature matrix (xem `DROP_COLS`). Encode vẫn chạy để giữ train/test sạch, nhưng các cột đó không đưa vào X.

In [9]:
# Maps imported từ build_features: DISAB_MAP, EDU_ORDER, IMD_ORDER
for df in [train, test]:
    df['disability']        = df['disability'].map(DISAB_MAP)
    df['highest_education'] = df['highest_education'].map(EDU_ORDER)
    df['imd_band_filled']   = df['imd_band_filled'].map(IMD_ORDER)

# Label encode target — fit trên train
labels = sorted(train['final_result'].unique())
LABEL_MAP = {l: i for i, l in enumerate(labels)}
print('Label map:', LABEL_MAP)

Label map: {'Fail': 0, 'Pass': 1, 'Withdrawn': 2}


---

### 4. Build feature matrices

In [10]:
DROP_COLS = {'gender', 'region', 'age_band', 'imd_band'}
# prediction_point không phải feature — không đưa vào X, lưu riêng để modeling tách theo mốc
FEATURE_COLS = (
    [c for c in STATIC_FEATURES + dynamic_features if c not in DROP_COLS]
    + ['imd_band_filled', 'imd_missing']
)

X_train = train[FEATURE_COLS].copy()
X_test  = test[FEATURE_COLS].copy()

y_train = train['final_result'].map(LABEL_MAP)
y_test  = test['final_result'].map(LABEL_MAP)

print('X_train:', X_train.shape, '| NaN:', X_train.isna().sum().sum())
print('X_test: ', X_test.shape,  '| NaN:', X_test.isna().sum().sum())
print('y_train:\n', y_train.value_counts().sort_index())
print('y_test:\n',  y_test.value_counts().sort_index())
print('\nFeatures:', FEATURE_COLS)

X_train: (142384, 15) | NaN: 0
X_test:  (35910, 15) | NaN: 0
y_train:
 final_result
0    40333
1    88536
2    13515
Name: count, dtype: int64
y_test:
 final_result
0    10192
1    22432
2     3286
Name: count, dtype: int64

Features: ['highest_education', 'disability', 'num_of_prev_attempts', 'studied_credits', 'total_clicks_filled', 'active_weeks_filled', 'avg_weekly_clicks_filled', 'num_due', 'num_submitted_filled', 'avg_score_filled', 'num_failed_filled', 'avg_days_early_filled', 'no_submission_despite_due', 'imd_band_filled', 'imd_missing']


In [11]:
out_dir = Path('../data/processed')
out_dir.mkdir(exist_ok=True)

X_train.to_parquet(out_dir / 'X_train.parquet', index=False)
X_test.to_parquet(out_dir  / 'X_test.parquet',  index=False)
y_train.to_frame().to_parquet(out_dir / 'y_train.parquet', index=False)
y_test.to_frame().to_parquet(out_dir  / 'y_test.parquet',  index=False)

# Lưu prediction_point riêng để modeling dùng tách dữ liệu theo từng mốc
train[['prediction_point']].to_parquet(out_dir / 'train_meta.parquet', index=False)
test[['prediction_point']].to_parquet(out_dir  / 'test_meta.parquet',  index=False)

print('Saved to data/processed/')
print('Label map:', LABEL_MAP)
print('Feature cols:', FEATURE_COLS)

Saved to data/processed/
Label map: {'Fail': 0, 'Pass': 1, 'Withdrawn': 2}
Feature cols: ['highest_education', 'disability', 'num_of_prev_attempts', 'studied_credits', 'total_clicks_filled', 'active_weeks_filled', 'avg_weekly_clicks_filled', 'num_due', 'num_submitted_filled', 'avg_score_filled', 'num_failed_filled', 'avg_days_early_filled', 'no_submission_despite_due', 'imd_band_filled', 'imd_missing']
